# Lazy Evaluation: Philosophy, Mechanics, and Optimization Benefits

**Module**: 04_execution_engine  
**Notebook**: 01_lazy_evaluation.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How lazy evaluation works in Apache Spark, why transformations do not compute immediately, and how actions trigger execution.
- **Why does this topic exist?**: Lazy evaluation is the prerequisite for Catalyst query optimization, Whole-Stage Code Generation, and fault-tolerant DAG lineage.
- **What problem does it solve?**: Beginners frequently misattribute slow runtimes to the wrong lines of code (e.g. blaming filter() when count() actually ran).
- **Where is it used in real-world Data Engineering?**: Preventing premature action execution in automated ETL pipelines.

---

## 2. Imperative vs Lazy Declarative Execution

```
Imperative Execution (Python / Pandas):
Step 1: df = read_csv()           -> Reads entire file from disk immediately!
Step 2: df = df[df.val > 100]     -> Allocates new in-memory DataFrame!
Step 3: df = df[['id','val']]    -> Allocates third DataFrame!
(Intermediate memory churn, no cross-step optimization)

Lazy Execution (Apache Spark):
Step 1: df = read_csv()           -> Appends'LogicalRelation'to Plan. Zero I/O!
Step 2: df = df.filter(val > 100) -> Appends'Filter'node. Zero I/O!
Step 3: df = df.select('id','val')-> Appends'Project'node. Zero I/O!
Step 4: df.count()                -> ACTION! Catalyst fuses steps 1-3 into ONE SCAN!
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Lazy_Evaluation_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Timing Benchmark: Transformation vs Action Execution

In [ ]:
# Measure time to define transformations
t0 = time.time()
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
transformed = (
    orders
    .filter("quantity > 2")
    .filter("unit_price > 500")
    .withColumn("revenue", F.col("quantity") * F.col("unit_price"))
    .groupBy("channel")
    .sum("revenue")
)
t_declaration = time.time() - t0

# Measure time to trigger terminal action
t0 = time.time()
result = transformed.collect()
t_action = time.time() - t0

print(f"Time to declare transformations: {t_declaration:.5f}s (Building AST only)")
print(f"Time to execute terminal Action:  {t_action:.5f}s (Actual execution across cluster)")

---

## Experiment: Proving Predicate Pushdown via Lazy Evaluation

- **1. What are we testing?**: Demonstrate that filtering after reading CSV is pushed down and fused into the scan rather than reading all rows into memory.
- **2. What do we expect?**: Physical plan shows Filter pushed directly above or inside FileScan.

In [ ]:
plan_df = orders.filter("status = 'Cancelled'").select("order_id", "status")
print("Execution plan showing operator fusion:")
plan_df.explain("formatted")

**3. What happened?**: Filter and Project operators are executed as a single fused WholeStageCodegen step.  
**4. Why did it happen?**: Catalyst inspects the full plan before generating execution bytecode.  
**5. Practical Engineering Lesson**: Because Spark is lazy, you do not need to manually optimize the order of select and filter operations.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Action vs Transformation
**Problem Statement**: Given 5 operations (filter, count, show, select, write), classify each as Lazy Transformation or Eager Action.  
**Input / Context**: List of operations

In [ ]:
# Exercise 1: Classify operations

### Exercise 2 (Intermediate): Measure Latency Across Multiple Actions
**Problem Statement**: Run count(), first(), and take(5) on an uncached transformed DataFrame and observe that each executes independently.  
**Input / Context**: orders DataFrame

In [ ]:
# Exercise 2: Multiple actions latency

### Exercise 3 (Advanced): Eliminate Redundant Action Execution
**Problem Statement**: Refactor a code snippet that calls count() and show() into a single action using persist().  
**Input / Context**: orders DataFrame

In [ ]:
# Exercise 3: Cache before multiple actions

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Action vs Transformation):
```python
ops = {
"filter":"Lazy Transformation",
"count":"Eager Action",
"show":"Eager Action",
"select":"Lazy Transformation",
"write":"Eager Action"
}
for op, cat in ops.items():
    print(f"{op:10}: {cat}")
```

#### Solution 2 (Measure Latency Across Multiple Actions):
```python
tf = orders.filter("quantity > 3").groupBy("status").count()
for act in ["count()","first()","take(5)"]:
    t0 = time.time()
    if act =="count()": tf.count()
    elif act =="first()": tf.first()
    else: tf.take(5)
    print(f"{act:10} executed in {time.time()-t0:.4f}s")
```

#### Solution 3 (Eliminate Redundant Action Execution):
```python
cached_df = orders.filter("status ='Completed'").groupBy("channel").count().persist()
print("Count from first action:", cached_df.count())
cached_df.show()
cached_df.unpersist()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Laziness enables Optimization**: Allows Catalyst to prune columns, reorder joins, and fuse operators.
- **Actions trigger Jobs**: Nothing moves across the network until an Action is invoked.
- **Debugging Misconceptions**: Slow lines in Python are almost always the terminal Action calls.

### Essential Technical Terms
- `Lazy Evaluation`: Deferring computation until result is requested by an Action.
- `AST`: Abstract Syntax Tree representing the logical query.
- `WholeStageCodegen`: Fusing multiple physical operations into a single bytecode loop.

### Common Engineering Mistakes
- **Sprinkling count() for progress logs**: Recomputes the entire upstream query from scratch on every call.
- **Blaming filter() for slow runtime**: filter() takes 0.001s; the subsequent action executes the work.

### Interview Questions on This Topic
- **Q: Why did Spark design transformations to be lazy?**
  - *A*: To allow the Catalyst optimizer to inspect the entire query tree, eliminate unused columns, push predicates, and generate optimized physical execution code.
- **Q: What triggers execution in Spark?**
  - *A*: An Action (e.g. collect, count, take, write).

### What to Study Next
Open 04_execution_engine/02_dag_and_execution_plans.ipynb to read physical and logical plans.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')